## Project Description: Next Word Prediction Using LSTM
#### Project Overview:

This project aims to develop a deep learning model for predicting the next word in a given sequence of words. The model is built using a Gated Recurrent Unit (GRU), which is well-suited for sequence prediction tasks. The project includes the following steps:

1- Data Collection: We use the WikiText Dataset as our dataset. This dataset provides clean and structured text suitable for language modeling.

2- Data Preprocessing: The text data is cleaned, tokenized, converted into sequences, and padded to ensure uniform input lengths. The sequences are then split into training and testing sets.

3- Model Building: A GRU-based model is constructed with an embedding layer, stacked GRU layers, and a dense output layer with a softmax activation function to predict the probability of the next word.

4- Model Training: The model is trained using the prepared sequences, with early stopping implemented to prevent overfitting. Early stopping monitors the validation loss and stops training when performance stops improving.

5- Model Evaluation: The model is evaluated using example sentences to test its ability to predict the next word effectively.

6- Deployment: The trained model can be integrated into an application (such as a web interface) to provide real-time next-word predictions based on user input.

In [1]:
## Data Collection (similar to NLTK Gutenberg)

!pip install datasets

from datasets import load_dataset

## load dataset
data = load_dataset("wikitext", "wikitext-2-raw-v1")

## extract raw text
text_data = " ".join(data['train']['text'])

## save to file
with open('wikitext.txt', 'w', encoding='utf-8') as file:
    file.write(text_data)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [60]:
import numpy as np
import re

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

## load dataset
with open('wikitext.txt','r', encoding='utf-8') as file:
    text = file.read().lower()

## cleaning
text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
text = re.sub(r'\s+', ' ', text)

## limit data (for colab)
text = text[:120000]
## tokenize
tokenizer = Tokenizer(num_words=1000)
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
print(total_words)

4549


In [61]:
tokenizer.word_index

{'the': 1,
 'of': 2,
 'and': 3,
 'in': 4,
 'to': 5,
 'a': 6,
 'was': 7,
 'for': 8,
 'with': 9,
 's': 10,
 'by': 11,
 'as': 12,
 'that': 13,
 'on': 14,
 'at': 15,
 'her': 16,
 'were': 17,
 'it': 18,
 'from': 19,
 'she': 20,
 'is': 21,
 'an': 22,
 'had': 23,
 'game': 24,
 'be': 25,
 'he': 26,
 'his': 27,
 'first': 28,
 'two': 29,
 'valkyria': 30,
 'little': 31,
 'tower': 32,
 'rock': 33,
 'team': 34,
 'which': 35,
 'this': 36,
 'its': 37,
 'one': 38,
 'during': 39,
 'but': 40,
 'their': 41,
 'barker': 42,
 'also': 43,
 'not': 44,
 'blue': 45,
 'jackets': 46,
 'arsenal': 47,
 'are': 48,
 'after': 49,
 'chronicles': 50,
 'building': 51,
 'columbus': 52,
 'they': 53,
 '1': 54,
 '2': 55,
 'who': 56,
 'season': 57,
 'time': 58,
 'or': 59,
 'all': 60,
 '3': 61,
 'while': 62,
 'n': 63,
 'year': 64,
 'there': 65,
 'has': 66,
 'blackie': 67,
 '4': 68,
 'games': 69,
 'into': 70,
 'would': 71,
 'made': 72,
 'arkansas': 73,
 'four': 74,
 'when': 75,
 'only': 76,
 'been': 77,
 'fernandez': 78,
 'unde

In [62]:
## 🔥 CREATE token_list HERE
token_list = tokenizer.texts_to_sequences([text])[0]

seq_length = 25
input_sequences = []

for i in range(seq_length, len(token_list)):
    n_gram_seq = token_list[i-seq_length:i+1]
    input_sequences.append(n_gram_seq)

In [63]:
input_sequences

[[30,
  50,
  151,
  567,
  89,
  30,
  61,
  50,
  469,
  61,
  716,
  30,
  2,
  1,
  337,
  61,
  717,
  5,
  12,
  30,
  50,
  151,
  338,
  718,
  21,
  6],
 [50,
  151,
  567,
  89,
  30,
  61,
  50,
  469,
  61,
  716,
  30,
  2,
  1,
  337,
  61,
  717,
  5,
  12,
  30,
  50,
  151,
  338,
  718,
  21,
  6,
  944],
 [151,
  567,
  89,
  30,
  61,
  50,
  469,
  61,
  716,
  30,
  2,
  1,
  337,
  61,
  717,
  5,
  12,
  30,
  50,
  151,
  338,
  718,
  21,
  6,
  944,
  225],
 [567,
  89,
  30,
  61,
  50,
  469,
  61,
  716,
  30,
  2,
  1,
  337,
  61,
  717,
  5,
  12,
  30,
  50,
  151,
  338,
  718,
  21,
  6,
  944,
  225,
  398],
 [89,
  30,
  61,
  50,
  469,
  61,
  716,
  30,
  2,
  1,
  337,
  61,
  717,
  5,
  12,
  30,
  50,
  151,
  338,
  718,
  21,
  6,
  944,
  225,
  398,
  288],
 [30,
  61,
  50,
  469,
  61,
  716,
  30,
  2,
  1,
  337,
  61,
  717,
  5,
  12,
  30,
  50,
  151,
  338,
  718,
  21,
  6,
  944,
  225,
  398,
  288,
  24],
 [61,
  50,
  469,


In [64]:
## Pad Sequences
max_sequence_len=max([len(x) for x in input_sequences])
max_sequence_len

26

In [65]:
input_sequences=np.array(pad_sequences(input_sequences,maxlen=max_sequence_len,padding='pre'))
input_sequences

array([[ 30,  50, 151, ..., 718,  21,   6],
       [ 50, 151, 567, ...,  21,   6, 944],
       [151, 567,  89, ...,   6, 944, 225],
       ...,
       [ 38,   2,   1, ...,   6,   9, 195],
       [  2,   1, 116, ...,   9, 195,  86],
       [  1, 116, 352, ..., 195,  86,  11]], dtype=int32)

In [66]:
##create predicitors and label
import tensorflow as tf
x,y=input_sequences[:,:-1],input_sequences[:,-1]

In [67]:
x

array([[ 30,  50, 151, ..., 338, 718,  21],
       [ 50, 151, 567, ..., 718,  21,   6],
       [151, 567,  89, ...,  21,   6, 944],
       ...,
       [ 38,   2,   1, ...,   4,   6,   9],
       [  2,   1, 116, ...,   6,   9, 195],
       [  1, 116, 352, ...,   9, 195,  86]], dtype=int32)

In [68]:
y

array([  6, 944, 225, ..., 195,  86,  11], dtype=int32)

In [69]:
# Split the data into training and testing sets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [70]:
# Define early stopping
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

In [71]:
import numpy as np
import re

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from sklearn.model_selection import train_test_split

In [74]:
## Imports
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

## Early stopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

## Define model (FIXED)
model = Sequential()

# ✅ Input layer (fixes "unbuilt" issue)
model.add(Input(shape=(max_sequence_len - 1,)))

# Embedding layer (no input_length warning)
model.add(Embedding(total_words, 64))

# GRU layers
model.add(GRU(128, return_sequences=True))
model.add(Dropout(0.2))

model.add(GRU(64))

# Output layer
model.add(Dense(total_words, activation='softmax'))

## Compile (IMPORTANT)
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

## Summary (now works properly)
model.summary()

## Train model (Colab safe)
history = model.fit(
    x_train, y_train,
    epochs=80,
    batch_size=32,
    validation_data=(x_test, y_test),
    #callbacks=[early_stopping],
    verbose=1
)

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 25, 64)         │       291,136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_14 (GRU)                    │ (None, 25, 128)        │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 25, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_15 (GRU)                    │ (None, 64)             │        37,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 4549)           │       295,685 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 698,565 (2.66 MB)

 Trainable params: 698,565 (2.66 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/80
395/395 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.1075 - loss: 6.1065 - val_accuracy: 0.0964 - val_loss: 5.8196
Epoch 2/80
395/395 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.1083 - loss: 5.6804 - val_accuracy: 0.0964 - val_loss: 5.7921
Epoch 3/80
395/395 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.1083 - loss: 5.6463 - val_accuracy: 0.0964 - val_loss: 5.7724
Epoch 4/80
395/395 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.1148 - loss: 5.5586 - val_accuracy: 0.1049 - val_loss: 5.6956
Epoch 5/80
395/395 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.1192 - loss: 5.4518 - val_accuracy: 0.1074 - val_loss: 5.6776
Epoch 6/80
395/395 ━━━━━━━━━━━━━━━━━━━━ 5s 10ms/step - accuracy: 0.1206 - loss: 5.3641 - val_accuracy: 0.1078 - val_loss: 5.6536
Epoch 7/80
395/395 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.1232 - loss: 5.2894 - val_accuracy: 0.1087 - val_loss: 5.6345
Epoch 8/80
395/395 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.1264 - loss: 5.2127 - val_accuracy

In [75]:
# Function to predict the next word
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) >= max_sequence_len:
        token_list = token_list[-(max_sequence_len-1):]  # Ensure the sequence length matches max_sequence_len-1
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=1)
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None

In [80]:
input_text="my Love was painted "
print(f"Input text:{input_text}")
max_sequence_len=model.input_shape[1]+1
next_word=predict_next_word(model,tokenizer,input_text,max_sequence_len)
print(f"Next Word PRediction:{next_word}")

Input text:my Love was painted 
Next Word PRediction:such


In [82]:
# =========================
# SAVE EVERYTHING
# =========================

import pickle
from tensorflow.keras.models import load_model

# 1. Save trained model
model.save("next_word_model.h5")

# 2. Save tokenizer
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

# 3. Save config (sequence length + vocab)
config = {
    "max_sequence_len": max_sequence_len,
    "total_words": total_words
}

with open("config.pkl", "wb") as f:
    pickle.dump(config, f)

print("✅ All files saved successfully!")

# =========================
# DOWNLOAD FILES (COLAB)
# =========================

from google.colab import files

files.download("next_word_model.h5")
files.download("tokenizer.pkl")
files.download("config.pkl")

✅ All files saved successfully!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import tensorflow as tf
print(tf.__version__)